In [1]:
# importing libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from sklearn.metrics import (
    accuracy_score ,
    precision_score ,
    recall_score ,
    f1_score ,
    average_precision_score , 
    roc_auc_score ,
    confusion_matrix ,
    classification_report, 
    ConfusionMatrixDisplay 
)

In [2]:
lgbm_123 = pd.read_csv("../outputs/oof/lightgbm_seed_123_oof.csv")
lgbm_2024 = pd.read_csv("../outputs/oof/lightgbm_seed_2024_oof.csv")
lgbm_777 = pd.read_csv("../outputs/oof/lightgbm_seed_777_oof.csv")

In [3]:
lgbm_seeds_df = pd.DataFrame({
    "isFraud" : lgbm_123["isFraud"] ,
    "seed_123" : lgbm_123["lightbgm_seed_123_oof"] ,
    "seed_2024" : lgbm_2024["lightbgm_seed_2024_oof"] ,
    "seed_777" : lgbm_777["lightbgm_seed_777_oof"] ,
})

In [4]:
lgbm_seeds_df["lgbm_seed_ensemble"] = (lgbm_seeds_df["seed_123"] + lgbm_seeds_df["seed_2024"] + lgbm_seeds_df["seed_777"])/3

In [5]:
mask = lgbm_seeds_df["lgbm_seed_ensemble"].notna()

y_oof = lgbm_seeds_df.loc[mask , "isFraud"]
lgbm_seeds_pred = lgbm_seeds_df.loc[mask , "lgbm_seed_ensemble"]
lgbm_seeds_roc_auc = roc_auc_score(y_oof , lgbm_seeds_pred)
lgbm_seeds_pr_auc = average_precision_score(y_oof , lgbm_seeds_pred)

print(f"LightGBM Seed Ensemble ROC-AUC = {lgbm_seeds_roc_auc:.4f}")
print(f"LightGBM Seed Ensemble PR-AUC = {lgbm_seeds_pr_auc:.4f}")

LightGBM Seed Ensemble ROC-AUC = 0.8951
LightGBM Seed Ensemble PR-AUC = 0.5492


In [6]:
xgb_123 = pd.read_csv("../outputs/oof/xgboost_seed_123_oof.csv")
xgb_2024 = pd.read_csv("../outputs/oof/xgboost_seed_2024_oof.csv")
xgb_777 = pd.read_csv("../outputs/oof/xgboost_seed_777_oof.csv")

In [7]:
np.array_equal(xgb_123["isFraud"].values , xgb_2024["isFraud"].values) , np.array_equal(xgb_123["isFraud"].values , xgb_777["isFraud"].values)

(True, True)

In [8]:
xgb_seeds_df = pd.DataFrame({
    "isFraud" : xgb_123["isFraud"] ,
    "seed_123" : xgb_123["xgboost_seed_123_oof"] ,
    "seed_2024" : xgb_2024["xgboost_seed_2024_oof"] ,
    "seed_777" : xgb_777["xgboost_seed_777_oof"] ,
})

In [9]:
xgb_seeds_df["xgb_seed_ensemble"] = (xgb_seeds_df["seed_123"] + xgb_seeds_df["seed_2024"] + xgb_seeds_df["seed_777"])/3

In [10]:
mask = xgb_seeds_df["xgb_seed_ensemble"].notna()

y_oof = xgb_seeds_df.loc[mask , "isFraud"]
xgb_seeds_pred = xgb_seeds_df.loc[mask , "xgb_seed_ensemble"]
xgb_seeds_roc_auc = roc_auc_score(y_oof , xgb_seeds_pred)
xgb_seeds_pr_auc = average_precision_score(y_oof , xgb_seeds_pred)

print(f"XGBoost Seed Ensemble ROC-AUC = {xgb_seeds_roc_auc:.4f}")
print(f"XGBoost Seed Ensemble PR-AUC = {xgb_seeds_pr_auc:.4f}")

XGBoost Seed Ensemble ROC-AUC = 0.8967
XGBoost Seed Ensemble PR-AUC = 0.5734


In [11]:
catboost_oof = pd.read_csv("../outputs/oof/catboost_oof.csv")

In [12]:
weighted_df = pd.DataFrame({
    "isFraud" : catboost_oof["isFraud"] , 
    "catboost" : catboost_oof["catboost_oof"] , 
    "lightgbm" : lgbm_seeds_df["lgbm_seed_ensemble"] , 
    "xgboost" : xgb_seeds_df["xgb_seed_ensemble"]
})

In [13]:
weighted_df.isna().sum()

isFraud         0
catboost    98425
lightgbm    98425
xgboost     98425
dtype: int64

In [14]:
weighted_df["equal_ensemble"] = (weighted_df["catboost"] + weighted_df["lightgbm"] + weighted_df["xgboost"])/3

In [15]:
mask = weighted_df["equal_ensemble"].notna()

y_oof = weighted_df.loc[mask , "isFraud"]
pred = weighted_df.loc[mask , "equal_ensemble"]

print("ROC-AUC : " , roc_auc_score(y_oof , pred))
print("PR-AUC : " , average_precision_score(y_oof , pred))

ROC-AUC :  0.9228609721692703
PR-AUC :  0.622659284844819


In [16]:
weights = [
    (0.50 , 0.30 , 0.20) , 
    (0.50 , 0.25 , 0.25) ,
    (0.60 , 0.25 , 0.15) ,
    (0.60 , 0.20 , 0.20) ,
    (0.40 , 0.35 , 0.25) , 
    (0.40 , 0.40 , 0.20) , 
    (0.45 , 0.35 , 0.20) ,    
]

results = []

for cat_w , lgb_w , xgb_w in weights:
    pred = (
        cat_w * weighted_df["catboost"] + lgb_w * weighted_df["lightgbm"] + xgb_w * weighted_df["xgboost"]
    )

    mask = pred.notna()
    roc_auc = roc_auc_score(weighted_df.loc[mask , "isFraud"] , pred.loc[mask])
    pr_auc = average_precision_score(weighted_df.loc[mask , "isFraud"] , pred.loc[mask])

    results.append({
        "Catboost" : cat_w , 
        "LightGBM" : lgb_w , 
        "XGBoost" : xgb_w ,
        "ROC-AUC" : roc_auc,
        "PR-AUC" : pr_auc
    })

In [17]:
results_df = pd.DataFrame(results)
print(results_df.sort_values("ROC-AUC" , ascending=False))

   Catboost  LightGBM  XGBoost   ROC-AUC    PR-AUC
1      0.50      0.25     0.25  0.923281  0.624942
3      0.60      0.20     0.20  0.923031  0.623346
0      0.50      0.30     0.20  0.922932  0.623695
4      0.40      0.35     0.25  0.922883  0.623359
6      0.45      0.35     0.20  0.922728  0.623045
2      0.60      0.25     0.15  0.922571  0.621850
5      0.40      0.40     0.20  0.922433  0.621768
